# JobSpy job-hunt wrapper (notebook)

Inspired by:

- [`JobSpy`](https://github.com/speedyapply/JobSpy)
- [`job-hunt-utill`](https://github.com/shubhamshelar/job-hunt-utill)

This notebook:

- Scrapes jobs from multiple boards via `python-jobspy` (Indeed, LinkedIn, ZipRecruiter, Google).
- Lets you configure **titles**, **locations**, **sites**, and **results per search**.
- Tracks **seen job URLs** in a CSV so you only get **new jobs** on each run.
- Saves raw + filtered CSVs under `data/raw/` and `data/output/`.

Run cells **top to bottom** whenever you want fresh jobs.

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import csv

import pandas as pd
from jobspy import scrape_jobs

# ------------------ CONFIG (like job-hunt-utill/config.py) ------------------

# Job titles to search for
TITLES = [
    "data analyst",
    "business analyst",
    "analytics engineer",
]

# Locations to search
LOCATIONS = [
    "Remote",
    "Pune, India",
    "Mumbai, India",
]

# Sites to scrape (JobSpy site_name)
SITES = ["linkedin", "indeed", "zip_recruiter", "google"]
# Glassdoor / Bayt / Naukri / BDJobs can be flaky/blocked; keep them off by default.

# Results per (title, location, site)
RESULTS_PER_SEARCH = 20

# Time window: last N hours (e.g. 24h or 24*7 for 7 days)
HOURS_OLD = 24

# Country for Indeed/Glassdoor (see JobSpy README)
COUNTRY_INDEED = "india"  # 'usa', 'india', etc.

# Directory structure (mimics job-hunt-utill/data layout)
DATA_DIR = Path("data")
RAW_DIR = DATA_DIR / "raw"
OUTPUT_DIR = DATA_DIR / "output"
SEEN_FILE = DATA_DIR / "seen_jobs.csv"

for d in (RAW_DIR, OUTPUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

print("Raw dir   :", RAW_DIR.resolve())
print("Output dir:", OUTPUT_DIR.resolve())
print("Seen file :", SEEN_FILE.resolve())

Raw dir   : W:\CodeBase\Resume-Projects\sumit-personal-site\job-search-api\jobspy-testing\data\raw
Output dir: W:\CodeBase\Resume-Projects\sumit-personal-site\job-search-api\jobspy-testing\data\output
Seen file : W:\CodeBase\Resume-Projects\sumit-personal-site\job-search-api\jobspy-testing\data\seen_jobs.csv


In [2]:
def load_seen_urls(path: Path = SEEN_FILE) -> set[str]:
    """Load previously seen job URLs from CSV (one column: job_url)."""
    if not path.exists():
        return set()
    try:
        df = pd.read_csv(path)
        if "job_url" not in df.columns:
            return set()
        return set(str(u) for u in df["job_url"].dropna().unique())
    except Exception as e:
        print("Warning: failed to load seen URLs:", e)
        return set()


def save_seen_urls(urls: set[str], path: Path = SEEN_FILE) -> None:
    """Persist seen job URLs to CSV."""
    if not urls:
        return
    df = pd.DataFrame(sorted(urls), columns=["job_url"])
    df.to_csv(path, index=False)
    print(f"Saved seen log with {len(urls)} URLs -> {path}")


def normalize_location(loc: str) -> str:
    """Normalize our presets to something JobSpy boards accept.
    E.g. "Remote – India" -> "Remote" (keep is_remote flag separately).
    """
    s = (loc or "").strip()
    lower = s.lower()
    if lower.startswith("remote"):
        return "Remote"
    return s

In [ ]:
def run_scrape(hours_old: int = HOURS_OLD) -> None:
    seen = load_seen_urls()
    print(f"Loaded {len(seen)} seen URLs")

    all_frames: list[pd.DataFrame] = []

    for title in TITLES:
        for loc_raw in LOCATIONS:
            loc = normalize_location(loc_raw)
            for site in SITES:
                kwargs = dict(
                    site_name=[site],
                    search_term=title,
                    location=loc,
                    results_wanted=RESULTS_PER_SEARCH,
                    hours_old=hours_old,
                    country_indeed=COUNTRY_INDEED,
                    verbose=1,
                )
                print("\n== Scraping ==", site, "|", title, "|", loc, "with:", kwargs)
                try:
                    df = scrape_jobs(**kwargs)
                except Exception as e:
                    print(f"[ERROR] {site} failed for {title} @ {loc} -> {e}")
                    continue

                if df is None or df.empty:
                    print(f"[INFO] {site}: no jobs returned for this combo.")
                    continue

                df["site"] = df.get("site", site)
                df["search_title"] = title
                df["search_location"] = loc_raw
                all_frames.append(df)
                print(f"[OK] {site}: {len(df)} rows")

    if not all_frames:
        print("\nNo jobs scraped.")
        return

    raw_df = pd.concat(all_frames, ignore_index=True)
    print(f"\nTotal scraped rows (before dedupe): {len(raw_df)}")

    # Save raw CSV
    ts = datetime.utcnow().strftime("%Y-%m-%d_%H-%M")
    raw_path = RAW_DIR / f"jobs_raw_{ts}.csv"
    raw_df.to_csv(raw_path, index=False, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\")
    print("Saved raw CSV ->", raw_path)

    # Filter unseen by job_url (job-hunt-utill style)
    url_col_candidates = [c for c in raw_df.columns if c.lower() in {"job_url", "url", "joburl"}]
    if not url_col_candidates:
        print("No job_url/url column found; cannot filter unseen. Skipping seen log.")
        return

    url_col = url_col_candidates[0]
    raw_df["job_url_norm"] = raw_df[url_col].astype(str)

    new_mask = ~raw_df["job_url_norm"].isin(seen)
    new_df = raw_df.loc[new_mask].copy()
    print(f"New (unseen) rows: {len(new_df)}")

    if len(new_df) == 0:
        print("No new jobs; seen log unchanged.")
        return

    # Save new-only CSV
    out_path = OUTPUT_DIR / f"jobs_new_{ts}.csv"
    cols = [c for c in ["site", "title", "company", "location", "job_url_norm"] if c in new_df.columns] or list(new_df.columns)
    new_df.to_csv(out_path, index=False, quoting=csv.QUOTE_NONNUMERIC, escapechar="\\")
    print("Saved filtered new jobs CSV ->", out_path)

    # Update seen log
    new_urls = set(new_df["job_url_norm"].dropna().unique())
    seen.update(new_urls)
    save_seen_urls(seen)


run_scrape()

Loaded 0 seen URLs

== Scraping == linkedin | data analyst | Remote with: {'site_name': ['linkedin'], 'search_term': 'data analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}


2026-03-11 19:09:14,636 - ERROR - JobSpy:LinkedIn - LinkedIn: HTTPSConnectionPool(host='www.linkedin.com', port=443): Max retries exceeded with url: /jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=data+analyst&location=Remote&distance=50&pageNum=0&start=0&f_TPR=r86400 (Caused by NameResolutionError("HTTPSConnection(host='www.linkedin.com', port=443): Failed to resolve 'www.linkedin.com' ([Errno 11001] getaddrinfo failed)"))
2026-03-11 19:09:14,638 - INFO - JobSpy:Linkedin - finished scraping


[INFO] linkedin: no jobs returned for this combo.

== Scraping == indeed | data analyst | Remote with: {'site_name': ['indeed'], 'search_term': 'data analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[ERROR] indeed failed for data analyst @ Remote -> HTTPSConnectionPool(host='apis.indeed.com', port=443): Max retries exceeded with url: /graphql (Caused by NameResolutionError("HTTPSConnection(host='apis.indeed.com', port=443): Failed to resolve 'apis.indeed.com' ([Errno 11001] getaddrinfo failed)"))

== Scraping == zip_recruiter | data analyst | Remote with: {'site_name': ['zip_recruiter'], 'search_term': 'data analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[ERROR] zip_recruiter failed for data analyst @ Remote -> failed to do request: Post "https://api.ziprecruiter.com/jobs-app/event": dial tcp: lookup api.ziprecruiter.com: no such host

== Scraping == google | 

2026-03-11 19:10:14,151 - ERROR - JobSpy:LinkedIn - LinkedIn: HTTPSConnectionPool(host='www.linkedin.com', port=443): Max retries exceeded with url: /jobs-guest/jobs/api/seeMoreJobPostings/search?keywords=data+analyst&location=Pune%2C+India&distance=50&pageNum=0&start=0&f_TPR=r86400 (Caused by NameResolutionError("HTTPSConnection(host='www.linkedin.com', port=443): Failed to resolve 'www.linkedin.com' ([Errno 11001] getaddrinfo failed)"))


[INFO] linkedin: no jobs returned for this combo.

== Scraping == indeed | data analyst | Pune, India with: {'site_name': ['indeed'], 'search_term': 'data analyst', 'location': 'Pune, India', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[ERROR] indeed failed for data analyst @ Pune, India -> HTTPSConnectionPool(host='apis.indeed.com', port=443): Max retries exceeded with url: /graphql (Caused by NameResolutionError("HTTPSConnection(host='apis.indeed.com', port=443): Failed to resolve 'apis.indeed.com' ([Errno 11001] getaddrinfo failed)"))

== Scraping == zip_recruiter | data analyst | Pune, India with: {'site_name': ['zip_recruiter'], 'search_term': 'data analyst', 'location': 'Pune, India', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[ERROR] zip_recruiter failed for data analyst @ Pune, India -> failed to do request: Post "https://api.ziprecruiter.com/jobs-app/event": dial tcp: lookup api.ziprecruiter.com: no such 

...

[INFO] google: no jobs returned for this combo.

== Scraping == linkedin | business analyst | Remote with: {'site_name': ['linkedin'], 'search_term': 'business analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[OK] linkedin: 20 rows

== Scraping == indeed | business analyst | Remote with: {'site_name': ['indeed'], 'search_term': 'business analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[OK] indeed: 5 rows

== Scraping == zip_recruiter | business analyst | Remote with: {'site_name': ['zip_recruiter'], 'search_term': 'business analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[INFO] zip_recruiter: no jobs returned for this combo.

== Scraping == google | business analyst | Remote with: {'site_name': ['google'], 'search_term': 'business analyst', 'location': 'Remote', 'results_wanted': 20, 'hours_old': 2

2026-03-11 19:12:57,809 - WARNING - JobSpy:Google - initial cursor not found, try changing your query or there was at most 10 results


[INFO] google: no jobs returned for this combo.

== Scraping == linkedin | business analyst | Pune, India with: {'site_name': ['linkedin'], 'search_term': 'business analyst', 'location': 'Pune, India', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[OK] linkedin: 20 rows

== Scraping == indeed | business analyst | Pune, India with: {'site_name': ['indeed'], 'search_term': 'business analyst', 'location': 'Pune, India', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[OK] indeed: 20 rows

== Scraping == zip_recruiter | business analyst | Pune, India with: {'site_name': ['zip_recruiter'], 'search_term': 'business analyst', 'location': 'Pune, India', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}
[INFO] zip_recruiter: no jobs returned for this combo.

== Scraping == google | business analyst | Pune, India with: {'site_name': ['google'], 'search_term': 'business analyst', 'location': 'Pune, Ind

2026-03-11 19:13:06,208 - WARNING - JobSpy:Google - initial cursor not found, try changing your query or there was at most 10 results


[INFO] google: no jobs returned for this combo.

== Scraping == linkedin | business analyst | Mumbai, India with: {'site_name': ['linkedin'], 'search_term': 'business analyst', 'location': 'Mumbai, India', 'results_wanted': 20, 'hours_old': 24, 'country_indeed': 'india', 'verbose': 1}


In [ ]:
from glob import glob

files = sorted(glob(str(OUTPUT_DIR / "jobs_new_*.csv")))
if not files:
    print("No jobs_new_*.csv files yet.")
else:
    latest = files[-1]
    print("Latest:", latest)
    df_latest = pd.read_csv(latest)
    cols = [c for c in ["site", "title", "company", "location", "job_url"] if c in df_latest.columns]
    display(df_latest[cols].head(20) if cols else df_latest.head(20))